# fastgit

> Use git from python, fast

fastgit wraps the `git` CLI. The base `Git` class turns every git subcommand into a method call and returns whatever git printed, and its `Repo` subclass adds live objects for the things git talks about: commits, refs, status, diffs, blame. Reprs mirror the terminal, so displaying an object shows roughly what the matching git command would have shown you.

> **NB**: If you are reading this in GitHub's readme, we recommend you instead read the much more nicely formatted [documentation format](https://AnswerDotAI.github.io/fastgit/) of this tutorial.

## Install

```sh
pip install fastgit
```

## Getting started

In [ ]:
from fastgit import *
from pathlib import Path
import tempfile, shutil

Everything below happens in a temporary directory, so it's safe to run anywhere. `Repo` (like `Git`) dispatches any attribute as a git subcommand: `r.init(b='main')` runs `git init -b main`, and the reply is the string git printed.

In [ ]:
td = tempfile.mkdtemp()
r = Repo(td)
r.init(b='main')

'Initialized empty Git repository in /tmp/tmp111odbt6/.git/'

Keyword arguments become flags: one-letter names get one dash (`b='main'` is `-b main`), longer names get two (`no_ff=True` is `--no-ff`; `True` means the flag takes no value), and `__=['path']` puts paths after `--`. When git fails, the method prints git's one-line complaint and returns None; pass `raise_exc=True` to get an exception instead.

## Commits

git stores snapshots: a commit points at a complete tree, and log, diff, and status are views computed from those snapshots. On `Repo`, `commit` returns the new head `Commit`, whose repr is its `--oneline` row. (Committing needs an identity, hence the `config` calls.)

In [ ]:
r.config('user.name', 'fastgit')
r.config('user.email', 'fastgit@example.com')
(r.d/'shop.txt').write_text('bread\nmilk\n')
r.add('.')
r.commit('start the list')

d05bd08 start the list

In [ ]:
(r.d/'shop.txt').write_text('bread\nmilk\neggs\n')
r.add('.')
c = r.commit('need eggs')
r.log()

ba8bf40 need eggs
d05bd08 start the list

`log` takes git's own range syntax and flags (`r.log('main..feat')`, `n=10`). `at` resolves any rev to a single `Commit`, and `c.parent` walks up the graph:

In [ ]:
r.at('HEAD~1')

d05bd08 start the list

## Diffs

Since a commit is a snapshot, a patch is a comparison between two of them, and `b = a + patch` means the patch is `b - a`. Subtraction returns a `Diff`, file rows shown `--stat`-style, with the full text one property away:

In [ ]:
c - c.parent

shop.txt | +1 -0
1 files changed, +1 -0

In [ ]:
print((c - c.parent).patch)

diff --git a/shop.txt b/shop.txt
index 26d3bde..5c4c692 100644
--- a/shop.txt
+++ b/shop.txt
@@ -1,2 +1,3 @@
 bread
 milk
+eggs


## Status

`status` compares HEAD, the index, and the working directory, including untracked files, shown as `git status -sb` would. The codes are porcelain v2's: `.M` is modified but unstaged, `M.` staged, `??` untracked:

In [ ]:
(r.d/'shop.txt').write_text('bread\nmilk\neggs\njam\n')
(r.d/'notes.txt').write_text('todo\n')
r.status

## main
.M shop.txt
?? notes.txt

In [ ]:
r.add('-A')
r.commit('add jam and notes')
r.status.clean

True

## Merges: a conflict is a status, not an error

Every op that changes the working tree (`merge`, `rebase`, `pull`, `stash`) returns the resulting `Status`, clean or conflicted. Nothing raises on conflict, because git considers a paused merge a normal state; you read the status to see where you stand. Let's manufacture a conflict:

In [ ]:
r.switch('-c', 'feat')
(r.d/'shop.txt').write_text('bread\nmilk\neggs\njam\nbutter\n')
r.add('.')
r.commit('feat: butter')
r.switch('main')
(r.d/'shop.txt').write_text('bread\nmilk\neggs\njam\ncheese\n')
r.add('.')
r.commit('main: cheese')
r.merge('feat')

## main
UU shop.txt
# merge in progress

The conflicted entry's three versions are readable as `:1:path` (base), `:2:path` (ours), and `:3:path` (theirs). `cat` reads them exactly, byte for byte:

In [ ]:
print(r.cat(':3:shop.txt'))

bread
milk
eggs
jam
butter



Resolution is ordinary git: write the file, `add`, `commit`. The two parents on the new head are the proof the merge concluded:

In [ ]:
(r.d/'shop.txt').write_text('bread\nmilk\neggs\njam\nbutter\ncheese\n')
r.add('.')
mc = r.commit('merge feat')
len(mc.parents)

2

## Blame and trace

`blame` maps each line to the commit that last touched it, and each row's `.commit` is the full handle. Our shopping list is now spread over five commits, two of them from different sides of the merge:

In [ ]:
r.blame('shop.txt')

d05bd08 (fastgit 2026-07-24 13:15   1) bread
d05bd08 (fastgit 2026-07-24 13:15   2) milk
ba8bf40 (fastgit 2026-07-24 13:15   3) eggs
df130c1 (fastgit 2026-07-24 13:15   4) jam
163d8a7 (fastgit 2026-07-24 13:15   5) butter
77975d6 (fastgit 2026-07-24 13:15   6) cheese

The `-L` range forms come as keywords: `lines=(start,end)`, `func='name'` (git's `:funcname` form, which finds a definition by name), and `regex=` for content matching. `trace` is `git log -L`, the history of a range: ordinary `Commits`, each carrying the `.patch` that changed it.

In [ ]:
(r.d/'prices.py').write_text('def total(xs):\n    return sum(xs)\n')
r.add('.')
r.commit('add total')
(r.d/'prices.py').write_text('def total(xs):\n    return round(sum(xs), 2)\n')
r.add('.')
r.commit('round totals')
r.blame('prices.py', func='total')

5d40b66 (fastgit 2026-07-24 13:15   1) def total(xs):
0723d5c (fastgit 2026-07-24 13:15   2)     return round(sum(xs), 2)

In [ ]:
t = r.trace('prices.py', func='total')
t

0723d5c round totals
5d40b66 add total

In [ ]:
print(t[0].patch)

diff --git a/prices.py b/prices.py
index 8ce3223..bc210a6 100644
--- a/prices.py
+++ b/prices.py
@@ -1,2 +1,2 @@
 def total(xs):
-    return sum(xs)
+    return round(sum(xs), 2)


## Remotes

A bare directory is a perfectly good remote, so none of this needs a network. `push` returns the current branch's refreshed `Ref`, and after `push -u` its repr carries the tracking bracket, the same confirmation you'd look for in a terminal:

In [ ]:
bare = Path(tempfile.mkdtemp())/'origin.git'
Git(bare.parent)('init', '--bare', '-b', 'main', bare.name)
r.remote('add', 'origin', str(bare))
r.push('-u', 'origin', 'main')

* main 0723d5c [origin/main] round totals

`Repo.clone` is a classmethod, since until it runs there's no repo to hold a handle on:

In [ ]:
r2 = Repo.clone(bare, Path(tempfile.mkdtemp())/'copy')
r2.log(n=3)

0723d5c round totals
5d40b66 add total
b64a211 merge feat

When the clone pushes a commit, our first checkout is behind. `fetch` moves the remote-tracking refs and returns the refreshed branches, so the gap shows immediately; `pull` closes it:

In [ ]:
r2.config('user.name', 'fastgit')
r2.config('user.email', 'fastgit@example.com')
(r2.d/'shop.txt').write_text('bread\n')
r2.add('.')
r2.commit('simplify radically')
r2.push()
r.fetch()

  feat 163d8a7 feat: butter
* main 0723d5c [origin/main: behind 1] round totals

In [ ]:
r.pull().clean

True

## Stashes

A stash entry is a real commit under the hood, addressed `stash@{n}`. `stash` returns the now-clean `Status`, and `pop` returns the status with your changes back:

In [ ]:
(r.d/'notes.txt').write_text('urgent scribble\n')
r.stash('scribble')
r.stashes

stash@{0}: On main: scribble

In [ ]:
r.stashes[0].pop()

## main...origin/main
.M notes.txt

## Learn more

The [Repo docs](https://AnswerDotAI.github.io/fastgit/repo.html) are the full literate source: refs and tags, rebase (including aborting one mid-conflict), stash details, and the parsing that backs it all. For LLM agents, `fastgit.skill` packages this API as a [pyskills](https://AnswerDotAI.github.io/pyskills/) module.

In [ ]:
#| hide
for p in (Path(td), bare.parent, r2.d.parent): shutil.rmtree(p)